**Tratamiento de nulos contextual:**

- No es lo mismo un `NaN` en "Courier Status" (puede implicar que no se envió) que en "Amount" (implica venta sin registrar). Podrías imputar con `fillna()` usando la **mediana por categoría de producto** para el monto.
- Demuestra que no imputás mecánicamente, sino que **entendés el dato**.

In [47]:
# 1. Importación de librerías y carga del dataset

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Carga del dataset
df_raw = pd.read_csv('01_Amazon Sale Report.csv')

# Mostrá las primeras 5 filas con .head() para inspeccionar visualmente.
print("Primeras 5 filas del dataset:\n", df_raw.head(5), "\n")

# Mostramos cuántas filas y columnas hay con .shape
print("Número de filas y columnas:\n", df_raw.shape)

C:\Users\Woofe\AppData\Local\Temp\ipykernel_13208\701237663.py:9: DtypeWarning: Columns (0: Unnamed: 22) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv('01_Amazon Sale Report.csv')


Primeras 5 filas del dataset:
    index             Order ID      Date                        Status  \
0      0  405-8078784-5731545  04-30-22                     Cancelled   
1      1  171-9198151-1101146  04-30-22  Shipped - Delivered to Buyer   
2      2  404-0687676-7273146  04-30-22                       Shipped   
3      3  403-9615377-8133951  04-30-22                     Cancelled   
4      4  407-1069790-7240320  04-30-22                       Shipped   

  Fulfilment Sales Channel  ship-service-level    Style              SKU  \
0   Merchant      Amazon.in           Standard   SET389   SET389-KR-NP-S   
1   Merchant      Amazon.in           Standard  JNE3781  JNE3781-KR-XXXL   
2     Amazon      Amazon.in          Expedited  JNE3371    JNE3371-KR-XL   
3   Merchant      Amazon.in           Standard    J0341       J0341-DR-L   
4     Amazon      Amazon.in          Expedited  JNE3671  JNE3671-TU-XXXL   

        Category  ... currency  Amount    ship-city   ship-state  \
0    

In [48]:
# 2. Auditoría de calidad de datos

# Tipos de datos, memoria, cuántos no-nulos por columna.
print("Informacion del dataset:\n")
print(df_raw.info(), "\n") 

# Conteo exacto de nulos por columna.
print("Conteo de valores nulos por columna:\n", df_raw.isnull().sum(), "\n")

# Cantidad de filas duplicadas exactas.
print("Número de filas duplicadas exactas:\n", df_raw.duplicated().sum(), "\n")

# Estadísticas básicas de columnas numéricas y categóricas.
print("Estadísticas de columnas numéricas:\n", df_raw.describe(include='all'), "\n")

Informacion del dataset:

<class 'pandas.DataFrame'>
RangeIndex: 128975 entries, 0 to 128974
Data columns (total 24 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   index               128975 non-null  int64  
 1   Order ID            128975 non-null  str    
 2   Date                128975 non-null  str    
 3   Status              128975 non-null  str    
 4   Fulfilment          128975 non-null  str    
 5   Sales Channel       128975 non-null  str    
 6   ship-service-level  128975 non-null  str    
 7   Style               128975 non-null  str    
 8   SKU                 128975 non-null  str    
 9   Category            128975 non-null  str    
 10  Size                128975 non-null  str    
 11  ASIN                128975 non-null  str    
 12  Courier Status      122103 non-null  str    
 13  Qty                 128975 non-null  int64  
 14  currency            121180 non-null  str    
 15  Amount             

## El dataset cuenta con 24 Columnas, de las cuales:

- 1 columna trata con valores true o false
- 2 columnas trata con valores numericos reales que pueden incluir decimales
- 2 columnas trata con valores numericos enteros
- 1 columna trata con objetos (duda)
- 18 columnas trata con textos


## Hay 10 de 24 columnas que contienen valores nulos:

1)  Courier Status:         6872 valores nulos -> Estado del envío con la empresa de mensajería.

2)  currency:               7795 valores nulos -> Moneda en la que se realizó la venta (texto).

3)  Amount:                 7795 valores nulos -> Monto total de la venta (número decimal).

4)  ship-city:                33 valores nulos -> Ciudad de destino del envío (texto)

5)  ship-state:               33 valores nulos -> Estado o provincia del destino (texto)

6)  ship-postal-code:         33 valores nulos -> Código postal del destino (texto o número)

7)  ship-country:             33 valores nulos -> País de destino del envío

8)  promotion-ids:         49153 valores nulos -> Identificadores de promociones o descuentos aplicados al pedido (texto)

9)  fulfilled-by:          89698 valores nulos -> Indica quién gestionó el envío (texto)

10) Unnamed: 22:           49050 valores nulos -> Esta es una columna fantasma muy común en pandas

### Conlusiones:
- 33 nulos en ship-*: Podrían ser pedidos cancelados, en proceso, o datos históricos incompletos
- Alto % nulos en promotion-ids: La mayoría de las ventas se hacen sin códigos promocionales
- Alto % nulos en fulfilled-by: Quizás solo aplica a ciertos tipos de envío o regiones
- De 89698 (fulfilled-by) envios nulos, hay 7795 (Amount) que son valores nulos de venta, de los cuales 6872 (Courier Status) de estados de envios que no se realizaron, por lo cual hay una diferencia entre valores nulos de venta y estados de envio que no se realizaron de 923 posibles pedidos que se perdieron.

## No hay filas duplicadas (no es necesaria la acción de eliminar duplicados)

## Estadísticas de columnas numéricas:

Analizando la columna 'Amount' de valores de venta, podemos decir que:
- mínimo: 0          → Posibles ventas canceladas, muestras gratis u errores
- media: 648.56      → Promedio saludable
- mediana: 605       → Similar a la media (distribución ligeramente asimétrica)
- máximo: 5,584      → Existen ventas de alto valor
- std: 281.21        → Los montos varían ±281 alrededor de la media (desviación estándar)

Analizando la columna 'ship-postal-code', podemos decir que:
- Los valores oscilan entre 110001 y 989898, consistentes con códigos postales de India.

## Resumen:

- El dataset tiene 128,975 registros y 24 columnas.
- No hay filas duplicadas.
- Se detectaron 10 columnas con valores nulos, siendo 'fulfilled-by' (89,698 nulos) y 'promotion-ids' (49,153 nulos) las más afectadas.
- La columna 'Amount' tiene 7,795 nulos, que representan el 6% de los registros.
- Las columnas 'ship-*' tienen solo 33 nulos (0.03%), por lo que se eliminarán esas filas.
- Las columnas 'Unnamed: 22' e 'index' son redundantes y se eliminarán en la limpieza.
- La columna 'Qty' es binaria (solo 0 y 1). Los valores 0 serán analizados en conjunto con 'Amount' y 'Status' para determinar si corresponden a pedidos cancelados o errores.
- Se identificó una inconsistencia de 923 registros donde 'Amount' es nulo pero 'Courier Status' no lo es.

In [49]:
# 3. Tratamiento Contextual de Nulos

# Copia limpia del dataset para trabajar sin modificar el original.
df_clean = df_raw.copy()
print("Número de filas y columnas en el dataset limpio:\n", df_clean.shape, "\n")

# Eliminación de columnas redundantes o irrelevantes.
df_clean = df_raw.drop(columns=['Unnamed: 22', 'index']) # Eliminar la columna fantasma y la columna 'index' si existe.
print("Número de filas y columnas después de eliminar columnas irrelevantes:\n", df_clean.shape, "\n")

df_clean = df_clean.dropna(subset=['ship-city', 'ship-state', 'ship-postal-code', 'ship-country']) # Eliminar filas con nulos en columnas de envío.
print("Número de filas y columnas después de eliminar filas con nulos en columnas de envío:\n", df_clean.shape, "\n")

df_clean['promotion-ids'] = df_clean['promotion-ids'].fillna('Sin promoción') # Rellenar nulos en 'promotion-ids' con 'Sin promoción'.
df_clean['fulfilled-by'] = df_clean['fulfilled-by'].fillna('Vendedor') # Rellenar nulos en 'fulfilled-by' con 'Vendedor'.

# Filtrar filas donde Courier Status es nulo
nulos_courier = df_clean[df_clean['Courier Status'].isnull()]
# Imprimir los valores que toma la columna 'Status' para las filas donde 'Courier Status' es nulo.
print("Filas donde Courier Status es nulo: \n", nulos_courier['Status'].value_counts(), "\n")
'''
Acá observamos que los valores que toma la columna 'Status' para las filas donde 'Courier Status' es nulo son:

1- Cancelled                        (para 6858 filas donde 'Courier Status' es nulo)
2- Shipped - Delivered to Buyer     (para 8 filas donde 'Courier Status' es nulo)
3- Shipped - Returned to Seller     (para 3 filas donde 'Courier Status' es nulo)

Esto nos da una pista de cómo podríamos rellenar los nulos en 'Courier Status' basándonos en el valor de 'Status'. 
'''

# Filtrar filas donde currency es nulo
nulos_currency = df_clean[df_clean['currency'].isnull()]
# Imprimir los valores que toma la columna 'Status' para las filas donde 'currency' es nulo.
print("Filas donde currency es nulo: \n", nulos_currency['Status'].value_counts(), "\n")
'''
Acá observamos que los valores que toma la columna 'Status' para las filas donde 'currency' es nulo son:

1- Cancelled                          (para 7564 filas donde 'currency' es nulo)
2- Shipped                            (para 208 filas donde 'currency' es nulo)
3- Shipped - Delivered to Buyer       (para 8 filas donde 'currency' es nulo)
4- Shipping                           (para 8 filas donde 'currency' es nulo)
5- Shipped - Returned to Seller       (para 3 filas donde 'currency' es nulo)
6- Pending                            (para 2 filas donde 'currency' es nulo)

Esto nos da una pista de cómo podríamos rellenar los nulos en 'currency' basándonos en el valor de 'Status'.
'''

'''
Ejemplo de cómo podríamos rellenar los nulos en 'Courier Status' basándonos en el valor de 'Status' uno por uno:

# Rellenar nulos en 'Courier Status' con 'Cancelado' si el 'Status' es 'Cancelled'.
df_clean.loc[df_clean['Courier Status'].isna() & (df_clean['Status'] == 'Cancelled'), 'Courier Status'] = 'Cancelado' 

# Rellenar nulos en 'Courier Status' con 'No registrado' si el 'Status' es 'Shipped - Delivered to Buyer'.
df_clean.loc[df_clean['Courier Status'].isna() & (df_clean['Status'] == 'Shipped - Delivered to Buyer'), 'Courier Status'] = 'No registrado' 

# Rellenar nulos en 'Courier Status' con 'No registrado' si el 'Status' es 'Shipped - Returned to Seller'.
df_clean.loc[df_clean['Courier Status'].isna() & (df_clean['Status'] == 'Shipped - Returned to Seller'), 'Courier Status'] = 'No registrado'

'''

# Versión usando listas y código más compacto para rellenar nulos en 'Courier Status' basándonos en el valor de 'Status':

# Rellenar nulos en 'Courier Status' con 'Cancelado' si el 'Status' es 'Cancelled'.
df_clean.loc[df_clean['Courier Status'].isna() & (df_clean['Status'] == 'Cancelled'), 'Courier Status'] = 'Cancelado'

# Rellenar nulos en 'Courier Status' con 'No registrado' si el 'Status' es 'Shipped - Delivered to Buyer' o 'Shipped - Returned to Seller'.
df_clean.loc[df_clean['Courier Status'].isna() & (df_clean['Status'].isin(['Shipped - Delivered to Buyer', 'Shipped - Returned to Seller'])), 'Courier Status'] = 'No registrado'

# Verificar que ya no hay nulos en 'Courier Status'
print("¿Quedan nulos en Courier Status?", df_clean['Courier Status'].isna().sum())

'''
Ejemplo de cómo podríamos rellenar los nulos en 'currency' basándonos en el valor de 'Status' uno por uno:

# Rellenar nulos en 'currency' con 'Cancelado' si el 'Status' es 'Cancelled'.
df_clean.loc[df_clean['currency'].isna() & (df_clean['Status'] == 'Cancelled'), 'currency'] = 'Cancelado'

# Rellenar nulos en 'currency' con 'INR' si el 'Status' es 'Shipped'.
df_clean.loc[df_clean['currency'].isna() & (df_clean['Status'] == 'Shipped'), 'currency'] = 'INR'

# Rellenar nulos en 'currency' con 'INR' si el 'Status' es 'Shipped - Delivered to Buyer'.
df_clean.loc[df_clean['currency'].isna() & (df_clean['Status'] == 'Shipped - Delivered to Buyer'), 'currency'] = 'INR'

# Rellenar nulos en 'currency' con 'INR' si el 'Status' es 'Shipping'.
df_clean.loc[df_clean['currency'].isna() & (df_clean['Status'] == 'Shipping'), 'currency'] = 'INR'

# Rellenar nulos en 'currency' con 'INR' si el 'Status' es 'Shipped - Returned to Seller'.
df_clean.loc[df_clean['currency'].isna() & (df_clean['Status'] == 'Shipped - Returned to Seller'), 'currency'] = 'INR'

# Rellenar nulos en 'currency' con 'INR' si el 'Status' es 'Pending'.
df_clean.loc[df_clean['currency'].isna() & (df_clean['Status'] == 'Pending'), 'currency'] = 'INR'

'''

# Versión usando listas y código más compacto para rellenar nulos en 'currency' basándonos en el valor de 'Status':

# Rellenar nulos en 'currency' con 'Cancelado' si el 'Status' es 'Cancelled'.
df_clean.loc[df_clean['currency'].isna() & (df_clean['Status'] == 'Cancelled'), 'currency'] = 'Cancelado'

# Rellenar nulos en 'currency' con 'INR' si el 'Status' es 'Shipped', 'Shipped - Delivered to Buyer', 'Shipping', 'Shipped - Returned to Seller' o 'Pending'.
df_clean.loc[df_clean['currency'].isna() & (df_clean['Status'].isin(['Shipped', 'Shipped - Delivered to Buyer', 'Shipping', 'Shipped - Returned to Seller', 'Pending'])), 'currency'] = 'INR'

# Verificar que ya no hay nulos
print("¿Quedan nulos en currency?", df_clean['currency'].isna().sum())

Número de filas y columnas en el dataset limpio:
 (128975, 24) 

Número de filas y columnas después de eliminar columnas irrelevantes:
 (128975, 22) 

Número de filas y columnas después de eliminar filas con nulos en columnas de envío:
 (128942, 22) 

Filas donde Courier Status es nulo: 
 Status
Cancelled                       6858
Shipped - Delivered to Buyer       8
Shipped - Returned to Seller       3
Name: count, dtype: int64 

Filas donde currency es nulo: 
 Status
Cancelled                       7564
Shipped                          208
Shipped - Delivered to Buyer       8
Shipping                           8
Shipped - Returned to Seller       3
Pending                            2
Name: count, dtype: int64 

¿Quedan nulos en Courier Status? 0
¿Quedan nulos en currency? 0


In [50]:
# Primero, asegurarnos de que podemos agrupar por categoría
print("¿Hay nulos en Category?", df_clean['Category'].isna().sum())
print("Categorías disponibles:", df_clean['Category'].unique(), "\n")

# Segundo, calcular el precio mediano para cada categoría de producto
mediana_por_categoria = df_clean.groupby('Category')['Amount'].median()
print("Mediana de Amount por categoría:")
print(mediana_por_categoria, "\n")

# Calcular mediana global (por si acaso)
mediana_global = df_clean['Amount'].median()

# Para cada categoría, rellenar sus nulos con su mediana
for categoria in mediana_por_categoria.index:
    mascara = (df_clean['Category'] == categoria) & (df_clean['Amount'].isna())
    df_clean.loc[mascara, 'Amount'] = mediana_por_categoria[categoria]

# Los nulos que queden (por categoría nula) los rellenamos con mediana global
df_clean['Amount'] = df_clean['Amount'].fillna(mediana_global)

print("Amount limpiado correctamente")
print(f"Última verificación - nulos restantes: {df_clean['Amount'].isna().sum()}")

¿Hay nulos en Category? 0
Categorías disponibles: <StringArray>
[          'Set',         'kurta', 'Western Dress',           'Top',
  'Ethnic Dress',        'Bottom',         'Saree',        'Blouse',
       'Dupatta']
Length: 9, dtype: str 

Mediana de Amount por categoría:
Category
Blouse           545.00
Bottom           345.00
Dupatta          305.00
Ethnic Dress     837.00
Saree            791.00
Set              788.00
Top              519.05
Western Dress    744.00
kurta            435.00
Name: Amount, dtype: float64 

Amount limpiado correctamente
Última verificación - nulos restantes: 0


## Resumen del Tratamiento de Nulos (Explicación Conceptual)

Hasta este punto, hemos aplicado un **tratamiento inteligente** a los valores faltantes del dataset. A continuación, explico qué hicimos y por qué, sin entrar en detalles de código.

---

### 1. Eliminamos lo que sobraba

**Columnas eliminadas:**
- `Unnamed: 22` → Era una columna vacía, sin información útil (un "fantasma" técnico).
- `index` → Era un número de fila repetido que no aportaba nada al análisis.

**Filas eliminadas:**
- Registros sin datos de ubicación (`ship-city`, `ship-state`, `ship-postal-code`, `ship-country`).
- **Solo 33 filas** (menos del 0.03% del total), un porcentaje tan pequeño que no afecta los resultados.

---

### 2. Rellenamos los vacíos con sentido

#### a) Promociones y envíos
- **`promotion-ids`** (códigos de descuento): cuando estaba vacío, lo marcamos como `"Sin promoción"`. Porque un valor faltante aquí simplemente significa que no se aplicó ninguna oferta.
- **`fulfilled-by`** (quien gestionó el envío): el vacío indicaba que lo manejó el **vendedor directamente**. Lo renombramos como `"Vendedor"` para que sea una categoría clara.

#### b) Estado del envío (`Courier Status`) y moneda (`currency`)
Aquí no rellenamos de forma automática. **Investigamos** qué estaba pasando con esos pedidos:

- Cruzamos los datos con la columna `Status` (estado del pedido).
- Si el pedido estaba **cancelado**, marcamos `Courier Status` como `"Cancelado"` y `currency` como `"Cancelado"`.
- Si el pedido estaba **entregado, enviado o pendiente**, asumimos que el sistema no registró correctamente esos datos, y los completamos con valores lógicos:
  - `Courier Status` → `"No registrado"`
  - `currency` → `"INR"` (rupia india, la única moneda que aparecía en el resto de los datos).

De esta forma, **no inventamos información falsa**, sino que interpretamos el contexto del negocio.

#### c) Monto de la venta (`Amount`)
Esta era la columna más delicada, porque el monto es fundamental para medir ingresos y rentabilidad.

- **Estrategia:** Rellenamos cada monto faltante con el **precio mediano** de productos de la **misma categoría**.
- **¿Por qué la mediana?** Porque es un valor más estable que el promedio (no se deja influenciar por precios extremadamente altos o bajos).
- **¿Por qué por categoría?** Porque una blusa no cuesta lo mismo que un saree o un kurta. Agrupar por tipo de producto es más justo y realista.

Como respaldo, si alguna categoría no tenía precios de referencia, usamos la mediana global de todas las ventas.

---

### 3. Verificamos que todo quedó limpio

Después de cada paso, comprobamos que los valores nulos hubieran desaparecido. Al final:

- `Courier Status`: sin nulos
- `currency`: sin nulos
- `Amount`: sin nulos
- `Category`: no tenía nulos (lo verificamos)
- Las columnas de ubicación ya no tienen nulos (eliminamos esas pocas filas)

---

### Conclusión de esta etapa

El dataset ahora es **consistente** y **confiable**. Los valores faltantes no fueron simplemente "borrados" o "rellenados con cualquier número", sino que **tomamos decisiones basadas en el comportamiento real del negocio**. Esto nos permite avanzar a la siguiente fase del análisis sin distorsiones.

A continuación, aplicaremos técnicas para detectar valores atípicos (outliers), unificar criterios en los textos y preparar los datos para responder las preguntas de negocio.

In [51]:
# 4. Calcular cuartiles
'''
Los cuartiles son tres valores (Q1, Q2, Q3) que dividen un conjunto de datos ordenados en cuatro partes iguales, representando 
el 25%, 50% y 75% de la muestra. Son medidas de posición que facilitan el análisis de la distribución de los datos.

Primer Cuartil (Q1): El 25% de los datos son menores o iguales a este valor.
Segundo Cuartil (Q2): La mediana. El 50% de los datos son menores o iguales a este valor.
Tercer Cuartil (Q3): El 75% de los datos son menores o iguales a este valor.
'''

Q1 = df_clean['Amount'].quantile(0.25) # Calcular primer cuartil (Q1)
Q3 = df_clean['Amount'].quantile(0.75) # El tercer cuartil (Q3)
IQR = Q3 - Q1 # Muestra dónde se concentra el 50% central de los datos. Es el "corazón" de los datos, donde se concentra la mayoría normal.

'''
El IQR mide la dispersión de la mitad central de los datos y se utiliza para identificar outliers(valores atípicos).

outliers: observaciones numéricas que se alejan significativamente del resto de los datos en una muestra. Son puntos 
anómalos, inusualmente altos o bajos, que no siguen el patrón general y pueden sesgar el análisis estadístico, especialmente 
la media y la desviación estándar.

En concluisión, es una forma de discriminar los extremos y centrarnos en el rango donde se encuentra la mayoría de los datos normales.
'''

# Calcular límite superior (solo nos interesan valores altos)
limite_superior = Q3 + 1.5 * IQR
# El limite inferior sería Q1 - 1.5 *IQR (pero no nos interesa)
# 1.5 * IQR regla estándar para identificar valores atípicos (outliers) en un conjunto de datos.
# Si un valor es mayor que Q3 + 1.5 * IQR o menor que Q1 - 1.5 * IQR, se considera un outlier.

# Identificar outliers
# Creamos una máscara booleana donde True indica que la fila es un outlier (Amount > limite_superior) y False indica que no lo es.
outliers_mask = df_clean['Amount'] > limite_superior # Solo nos interesan los outliers altos, por eso comparamos con el límite superior.

# Contar cuántos outliers hay y qué porcentaje representan respecto al total de filas.
cantidad_outliers = outliers_mask.sum()
total_filas = len(df_clean)
porcentaje_outliers = (cantidad_outliers / total_filas) * 100

# Mostrar resultados
print("Detección de Outliers en 'Amount':\n")
print(f"Q1 (primer cuartil): {Q1:.2f}")
print(f"Q3 (tercer cuartil): {Q3:.2f}")
print(f"IQR (rango intercuartil): {IQR:.2f}")
print(f"Límite superior: {limite_superior:.2f}")
print(f"\nCantidad de outliers: {cantidad_outliers}")
print(f"Porcentaje de outliers: {porcentaje_outliers:.2f}%")

# Decisión según el porcentaje
if porcentaje_outliers < 2:
    print(f"\nDecisión: {porcentaje_outliers:.2f}% es menor al 2%. Se ELIMINAN los outliers.")
    df_clean = df_clean[~outliers_mask]  # Si dejamos [outliers_mask] dejamos la mascara de verdadero/falso como esta, al usar [~outliers_mask] invertimos la mascara y conservamos solo filas normales.
else:
    print(f"\nDecisión: {porcentaje_outliers:.2f}% es mayor o igual al 2%. Se CAPEAN los outliers.")
    df_clean.loc[outliers_mask, 'Amount'] = limite_superior  # Reducir al límite

# Verificar que ya no hay outliers
outliers_restantes = (df_clean['Amount'] > limite_superior).sum()
print(f"\nVerificación - Outliers restantes: {outliers_restantes}")
print(f"Nuevo tamaño del dataset: {df_clean.shape[0]} filas y {df_clean.shape[1]} columnas")

Detección de Outliers en 'Amount':

Q1 (primer cuartil): 437.14
Q3 (tercer cuartil): 788.00
IQR (rango intercuartil): 350.86
Límite superior: 1314.29

Cantidad de outliers: 3148
Porcentaje de outliers: 2.44%

Decisión: 2.44% es mayor o igual al 2%. Se CAPEAN los outliers.

Verificación - Outliers restantes: 0
Nuevo tamaño del dataset: 128942 filas y 22 columnas


## 4. Eliminación/Capeamiento de Outliers en 'Amount'

Los outliers son valores extremadamente altos o bajos que pueden distorsionar el análisis de LTV y rentabilidad. Aplicaremos el método del Rango Intercuartil (IQR) para identificarlos.

### 4.1 Cálculo de límites
- Q1 = primer cuartil (25% de los datos)
- Q3 = tercer cuartil (75% de los datos)
- IQR = Q3 - Q1
- Límite superior = Q3 + 1.5 * IQR
- Límite inferior = Q1 - 1.5 * IQR (no es necesario)

### 4.2 Identificación de outliers
- Se consideran outliers los valores de `Amount` que superen el límite superior.

### 4.3 Decisión
- Analizamos cuántos outliers hay en Amount. Si son menos del 1-2%, podemos eliminarlos sin miedo. Si son más, mejor capearlos.

In [52]:
# 5. Normalización de Strings (Textos)

# Seleccionar automáticamente todas las columnas de tipo texto
columnas_texto = df_clean.select_dtypes(include=['object', 'string']).columns.tolist()

# Normalización de textos: minúsculas y sin espacios
print("Columnas a normalizar:", columnas_texto)

# Aplicar limpieza de texto: minúsculas y sin espacios
for col in columnas_texto:

    # Verificar que la columna existe en el dataset limpio y que es de tipo texto (object) antes de aplicar la normalización.
    if col in df_clean.columns and df_clean[col].dtype in ['object', 'string']:

        # Contar valores únicos antes de normalizar para comparar después.
        original_unicos = df_clean[col].nunique() 

        # El método .str.strip() elimina espacios en blanco al inicio y al final de cada valor, mientras que .str.lower() convierte 
        # todo el texto a minúsculas. 
        # Esto ayuda a unificar los valores y reducir la cantidad de categorías únicas causadas por diferencias de formato.
        df_clean[col] = df_clean[col].str.strip().str.lower() 

        # Contar valores únicos después de normalizar para ver el impacto de la limpieza.
        nuevos_unicos = df_clean[col].nunique()

        print(f"\n'{col}': {original_unicos} valores únicos → {nuevos_unicos} después de normalizar")

print("\nNormalización de textos completada.")

Columnas a normalizar: ['Order ID', 'Date', 'Status', 'Fulfilment', 'Sales Channel ', 'ship-service-level', 'Style', 'SKU', 'Category', 'Size', 'ASIN', 'Courier Status', 'currency', 'ship-city', 'ship-state', 'ship-country', 'promotion-ids', 'fulfilled-by']

'Order ID': 120350 valores únicos → 120350 después de normalizar

'Date': 91 valores únicos → 91 después de normalizar

'Status': 13 valores únicos → 13 después de normalizar

'Fulfilment': 2 valores únicos → 2 después de normalizar

'Sales Channel ': 2 valores únicos → 2 después de normalizar

'ship-service-level': 2 valores únicos → 2 después de normalizar

'Style': 1377 valores únicos → 1377 después de normalizar

'SKU': 7195 valores únicos → 7195 después de normalizar

'Category': 9 valores únicos → 9 después de normalizar

'Size': 11 valores únicos → 11 después de normalizar

'ASIN': 7190 valores únicos → 7190 después de normalizar

'Courier Status': 5 valores únicos → 5 después de normalizar

'currency': 2 valores únicos → 2 

### Resultados de la normalización de textos

La normalización (conversión a minúsculas y eliminación de espacios) tuvo los siguientes efectos:

- **Columnas sin cambios:** `Order ID`, `SKU`, `ASIN`, `Status`, `Category`, etc. Ya estaban limpias o son códigos únicos.
- **Columnas con mejora significativa:**
  - `ship-city`: se redujo de 8,955 a 7,297 valores únicos (-1,658). Se unificaron ciudades escritas con diferencias de mayúsculas.
  - `ship-state`: se redujo de 69 a 47 valores únicos (-22). Se unificaron estados escritos de forma inconsistente.

✅ La normalización fue exitosa: se redujo la redundancia sin perder información.

In [53]:
# 6. Convertir 'Date' a datetime
df_clean['Date'] = pd.to_datetime(df_clean['Date'], format='%m-%d-%y', errors='coerce')

# Verificar cuántas fechas no se pudieron convertir
# Contamos las fechas nulas porque se dejan como NaT (Not a Time) cuando no se pueden convertir correctamente.
fechas_nulas = df_clean['Date'].isna().sum() 

if fechas_nulas > 0:
    print(f"Atención: {fechas_nulas} fechas no se pudieron convertir (se dejaron como NaT)")
else:
    print("'Date' convertida correctamente")

# Convertir 'B2B' a booleano (ya debería ser bool, pero validamos)
if df_clean['B2B'].dtype != 'bool':
    df_clean['B2B'] = df_clean['B2B'].astype(bool)
    print("'B2B' convertida a booleano")
else:
    print("'B2B' ya era booleano")

# Verificar tipos finales
print(df_clean.dtypes)

'Date' convertida correctamente
'B2B' ya era booleano
Order ID                         str
Date                  datetime64[us]
Status                           str
Fulfilment                       str
Sales Channel                    str
ship-service-level               str
Style                            str
SKU                              str
Category                         str
Size                             str
ASIN                             str
Courier Status                   str
Qty                            int64
currency                         str
Amount                       float64
ship-city                        str
ship-state                       str
ship-postal-code             float64
ship-country                     str
promotion-ids                    str
B2B                             bool
fulfilled-by                     str
dtype: object


In [ ]:
# 7. Crear variables a partir de la fecha
df_clean['Mes'] = df_clean['Date'].dt.month # De la columna 'Date' se extrae el número de mes (1 a 12) y se guarda en la nueva columna 'Mes'.
df_clean['Dia_Semana'] = df_clean['Date'].dt.dayofweek # De la columna 'Date' se extrae el día de la semana (0=lunes a 6=domingo) y se guarda en la nueva columna 'Dia_Semana'.
df_clean['Es_Finde'] = (df_clean['Dia_Semana'] >= 5).astype(bool) # De la columna 'Dia_Semana' se extrae True si es sábado o domingo (5 o 6) y se guarda en la nueva columna 'Es_Finde'.

print("✅ Variables creadas:")
print(f"   - 'Mes': número de mes (1 a 12)")
print(f"   - 'Dia_Semana': día de la semana (0=lunes a 6=domingo)")
print(f"   - 'Es_Finde': True si es sábado o domingo, False en caso contrario")

# Mostrar primeras filas para verificar
print("\nPrimeras 5 filas con las nuevas variables:")
print(df_clean[['Date', 'Mes', 'Dia_Semana', 'Es_Finde']].head())


=== FEATURE ENGINEERING ===

✅ Variables creadas:
   - 'Mes': número de mes (1 a 12)
   - 'Dia_Semana': día de la semana (0=lunes a 6=domingo)
   - 'Es_Finde': True si es sábado o domingo, False en caso contrario

Primeras 5 filas con las nuevas variables:
        Date  Mes  Dia_Semana  Es_Finde
0 2022-04-30    4           5      True
1 2022-04-30    4           5      True
2 2022-04-30    4           5      True
3 2022-04-30    4           5      True
4 2022-04-30    4           5      True


In [55]:
# 7. Guardar también en CSV (por si necesitás abrirlo en Excel)
df_clean.to_csv('df_clean.csv', index=False)

# 8.3 Resumen final
print("\n=== RESUMEN FINAL DEL HITO 2 ===")
print(f"Dataset original: 128,975 filas")
print(f"Dataset limpio: {df_clean.shape[0]:,} filas x {df_clean.shape[1]} columnas")


=== RESUMEN FINAL DEL HITO 2 ===
Dataset original: 128,975 filas
Dataset limpio: 128,942 filas x 25 columnas
